# Four-run prediction demo

This walkthrough loads the saved TF-IDF baseline and three DistilBERT checkpoints, then predicts several unseen news-style sentences with every run.

## Run training once (only if needed)

The demo needs the saved TF-IDF artifact and three DistilBERT checkpoints. Run the next cell only if those files are missing or you want to retrain. If you already ran `train.py`, skip it because training takes time.

In [ ]:
# Run only if the saved model artifacts are missing or need to be regenerated.
!cd .. && python scripts/train.py

In [2]:
from pathlib import Path
import pickle

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']
MODELS_DIR = Path('../outputs/models')
required = [MODELS_DIR / 'tfidf_logistic.pkl', *(MODELS_DIR / f'distilbert_{fraction}pct' for fraction in (10, 25, 100))]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing saved artifacts. Run `python ../scripts/train.py` from this notebooks directory first.\n' + '\n'.join(missing))

In [3]:
with (MODELS_DIR / 'tfidf_logistic.pkl').open('rb') as file:
    baseline = pickle.load(file)

bert_runs = {}
for fraction in (10, 25, 100):
    checkpoint = MODELS_DIR / f'distilbert_{fraction}pct'
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to('cpu').eval()
    bert_runs[f'DistilBERT ({fraction}% data)'] = (tokenizer, model)

RUN_NAMES = ['TF-IDF + Logistic Regression', *bert_runs.keys()]
print('Loaded:', ', '.join(RUN_NAMES))

Loaded: TF-IDF + Logistic Regression, DistilBERT (10% data), DistilBERT (25% data), DistilBERT (100% data)


In [4]:
def predict_all(text):
    rows = []
    probabilities = baseline['model'].predict_proba(baseline['vectorizer'].transform([text]))[0]
    label_id = int(probabilities.argmax())
    rows.append({'run': 'TF-IDF + Logistic Regression', 'prediction': LABELS[label_id], 'confidence': probabilities[label_id]})

    for name, (tokenizer, model) in bert_runs.items():
        encoded = tokenizer(text, truncation=True, max_length=128, return_tensors='pt')
        with torch.no_grad():
            probabilities = torch.softmax(model(**encoded).logits[0], dim=0).numpy()
        label_id = int(probabilities.argmax())
        rows.append({'run': name, 'prediction': LABELS[label_id], 'confidence': probabilities[label_id]})
    return pd.DataFrame(rows)

def demo_sentences(sentences):
    predictions = []
    for sentence in sentences:
        frame = predict_all(sentence)
        frame.insert(0, 'sentence', sentence)
        predictions.append(frame)
    return pd.concat(predictions, ignore_index=True)

## Example sentences

Edit or add sentences to test your own examples. The `consensus` column reports the majority prediction across the four runs.

In [5]:
sentences = [
    'The central bank raised interest rates after inflation remained above its target.',
    'The national team won the championship after scoring in extra time.',
    'Researchers unveiled a battery that charges electric vehicles in minutes.',
    'Leaders met to negotiate a ceasefire after weeks of fighting at the border.',
    'Shares of the retailer rose after it reported stronger quarterly earnings.',
    'The space agency launched a telescope to study distant galaxies.',
]

predictions = demo_sentences(sentences)
consensus = (predictions.groupby('sentence')['prediction']
             .agg(lambda values: values.mode().iat[0])
             .rename('consensus'))
display(predictions.join(consensus, on='sentence').style.format({'confidence': '{:.1%}'}))

,sentence,run,prediction,confidence,consensus
0,The central bank raised interest rates after inflation remained above its target.,TF-IDF + Logistic Regression,Business,61.6%,Business
1,The central bank raised interest rates after inflation remained above its target.,DistilBERT (10% data),Business,76.2%,Business
2,The central bank raised interest rates after inflation remained above its target.,DistilBERT (25% data),Business,74.9%,Business
3,The central bank raised interest rates after inflation remained above its target.,DistilBERT (100% data),Business,90.4%,Business
4,The national team won the championship after scoring in extra time.,TF-IDF + Logistic Regression,Sports,70.0%,Sports
5,The national team won the championship after scoring in extra time.,DistilBERT (10% data),Sports,90.3%,Sports
6,The national team won the championship after scoring in extra time.,DistilBERT (25% data),Sports,94.1%,Sports
7,The national team won the championship after scoring in extra time.,DistilBERT (100% data),Sports,89.4%,Sports
8,Researchers unveiled a battery that charges electric vehicles in minutes.,TF-IDF + Logistic Regression,Sci/Tech,34.5%,Sci/Tech
9,Researchers unveiled a battery that charges electric vehicles in minutes.,DistilBERT (10% data),Sci/Tech,66.9%,Sci/Tech


In [6]:
agreement = (predictions.groupby('sentence')['prediction']
             .agg(lambda values: values.nunique())
             .rename('different_predictions')
             .to_frame())
agreement['all_four_agree'] = agreement['different_predictions'].eq(1)
agreement

,different_predictions,all_four_agree
sentence,,
Leaders met to negotiate a ceasefire after weeks of fighting at the border.,1,True
Researchers unveiled a battery that charges electric vehicles in minutes.,1,True
Shares of the retailer rose after it reported stronger quarterly earnings.,1,True
The central bank raised interest rates after inflation remained above its target.,1,True
The national team won the championship after scoring in extra time.,1,True
The space agency launched a telescope to study distant galaxies.,1,True
